# LSTM Sequence Models — AI Engineering Interview Prep

Covers: vanilla RNN, LSTM gates, sequence classification and time series prediction with PyTorch.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cpu')
print(f"PyTorch {torch.__version__}")

## 1. LSTM Gate Equations (from scratch)

In [ ]:
class LSTMCell:
    """Single LSTM cell — shows the gate equations explicitly."""
    def __init__(self, input_size, hidden_size):
        # Combined weight matrix [W_f, W_i, W_g, W_o]
        self.W_xh = np.random.randn(input_size, 4 * hidden_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, 4 * hidden_size) * 0.01
        self.b    = np.zeros(4 * hidden_size)
        self.h = hidden_size

    def forward(self, x, h_prev, c_prev):
        gates = x @ self.W_xh + h_prev @ self.W_hh + self.b
        f, i, g, o = np.split(gates, 4, axis=-1)

        f = 1 / (1 + np.exp(-f))   # Forget gate
        i = 1 / (1 + np.exp(-i))   # Input gate
        g = np.tanh(g)              # Candidate cell state
        o = 1 / (1 + np.exp(-o))   # Output gate

        c = f * c_prev + i * g     # Cell state update
        h = o * np.tanh(c)         # Hidden state
        return h, c, {"f": f, "i": i, "g": g, "o": o}

cell = LSTMCell(input_size=5, hidden_size=4)
x     = np.random.randn(1, 5)
h0    = np.zeros((1, 4))
c0    = np.zeros((1, 4))
h1, c1, gates = cell.forward(x, h0, c0)
print("Gate values:")
for g, v in gates.items():
    print(f"  {g} (gate): {v.round(4)}")
print(f"h1: {h1.round(4)}")
print(f"c1: {c1.round(4)}")

## 2. Sequence Classification with PyTorch LSTM

In [ ]:
# Synthetic task: classify if a sequence has more positives than negatives
def make_dataset(n=1000, seq_len=20, noise=0.2):
    X = np.random.randn(n, seq_len, 1).astype(np.float32)
    y = (X[:, :, 0].sum(axis=1) > 0).astype(np.int64)
    return torch.tensor(X), torch.tensor(y)

X_all, y_all = make_dataset(n=2000)
split = 1600
X_tr, y_tr = X_all[:split], y_all[:split]
X_te, y_te = X_all[split:], y_all[split:]

from torch.utils.data import TensorDataset, DataLoader
train_ds = TensorDataset(X_tr, y_tr)
test_ds  = TensorDataset(X_te, y_te)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=256)

print(f"Train: {len(train_ds)}, Test: {len(test_ds)}")
print(f"Input shape: {X_tr.shape}  (batch, seq_len, features)")

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2, num_classes=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.norm   = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # out: (batch, seq_len, hidden); (h_n, c_n)
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = self.norm(out[:, -1, :])  # use last timestep
        return self.fc(self.dropout(last_hidden))

model = LSTMClassifier(input_size=1, hidden_size=32, num_layers=2)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_accs, test_accs = [], []
for epoch in range(15):
    model.train()
    correct, total = 0, 0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        correct += (out.argmax(1) == yb).sum().item()
        total += len(yb)
    train_accs.append(correct / total)

    model.eval()
    with torch.no_grad():
        all_preds = [model(xb).argmax(1) for xb, _ in test_dl]
        all_preds = torch.cat(all_preds)
        test_acc = (all_preds == y_te).float().mean().item()
    test_accs.append(test_acc)

    if epoch % 3 == 0:
        print(f"Epoch {epoch+1:2d}: train_acc={train_accs[-1]:.4f}, test_acc={test_acc:.4f}")

plt.plot(train_accs, label='Train'); plt.plot(test_accs, label='Test')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('LSTM Training'); plt.legend()
plt.show()

## Interview Tips

- **LSTM vs GRU**: LSTM has 4 gates (f, i, g, o) and separate cell state; GRU has 2 gates, no cell state. GRU trains faster with similar performance.
- **Vanishing gradient in RNNs**: fixed by LSTM's forget gate (can carry gradients over many timesteps).
- **batch_first=True**: input shape (batch, seq, features) — more natural. Default is (seq, batch, features).
- **Bidirectional LSTM**: processes sequence both forward and backward; doubles hidden_size in output.
- **Gradient clipping**: `clip_grad_norm_(params, max_norm)` — crucial for RNN stability.
- **When to use LSTM vs Transformer**: LSTMs for short sequences, streaming, embedded devices; Transformers for longer sequences with enough data.